# Alloy — Experiment Campaign

This notebook is the **entry point** for running a full Alloy evaluation campaign on Grid5000.
It provisions the cluster, generates the experiment matrix, runs each configuration via
[papermill](https://papermill.readthedocs.io) using `single_experiment.ipynb` as the per-run
template, and finally generates plots.

> **Do not run `single_experiment.ipynb` directly.** Its parameters are injected automatically
> by this notebook for each configuration.

## 0. Imports

In [ ]:
import os
cur_dir=os.getcwd() # save current directory to save the generated CSV files

## 1. Infrastructure Provisioning

Reserves nodes on Grid5000, installs Docker Swarm, sets up the overlay network,
deploys the monitoring stack (Prometheus/cAdvisor), and uploads the Docker images
built by `build.sh`.

**Parameters to adjust before running:**
- `cluster` — Grid5000 cluster name (e.g. `"gros"`, `"ecotype"`)
- `max_parallelism` — maximum parallelism level to benchmark;
  reserves `max_parallelism * 2 + 2` nodes
  (1 manager/injector, 1 job manager, `max_parallelism` task managers, `max_parallelism` Kafka brokers)
- `walltime` — Grid5000 job duration

In [ ]:
import enoslib as en
import pandas as pd
import sys
sys.path.insert(0, "../infra/grid5000")
job_name = "alloy-tests"

import init
role_name="control"
pattern_manager="control[0]"
cluster = "gros"

max_parallelism = 2

conf = (en.G5kConf.from_settings(job_name=job_name, walltime="2:00:00", job_type=[])
        .add_machine(roles=["manager", "control", "injector"], cluster=cluster, nodes=1)
        .add_machine(roles=["control","jobmanager"], cluster=cluster, nodes=1)
        .add_machine(roles=["control","taskmanager"], cluster=cluster, nodes=max_parallelism))

for i in range(max_parallelism):
    conf.add_machine(roles=["control",f"kafka{i+1}"], cluster=cluster, nodes=1)

local_conf = ( 
    conf.finalize()
)

provider = en.G5k(local_conf)
roles, networks = provider.init()

display(roles)
xp = init.AlloyG5KExperimentation(roles, pattern_manager)

xp.init_docker_swarm()
xp.init_docker_nodes()
with en.actions(roles=roles) as a:
    a.apt(name="python3-pip", state="present", update_cache="true")
    a.pip(name="docker<7.1.0", state="present")
    a.pip(name="requests<2.32", state="present")
xp.init_docker_network()
xp.init_monitoring()
xp.load_local_images(os.path.join(cur_dir, "../images"))

## 2. Experiment Matrix

Generates the full list of (query, throughput, parallelism) configurations to run
and writes them to `experiments/<date>/<uuid>/experiment.csv`.
Each row gets a unique UUID and a target output directory.

The throughput values are scaled per query to account for the mix of Nexmark event types:
- Q1, Q2, Q3: `throughput * 2` (bid-heavy queries)
- Q8: `throughput * 3` (multi-source join)
- Others: `throughput` as-is

**Adjust `queries`, `parallelisms`, `throughputs`, and `duration` to change the campaign scope.**

In [ ]:
import io
import uuid
import datetime
import pathlib

notebook_id = str(uuid.uuid4()) # Generate ID for current experiment

queries = [1, 2, 3, 5, 8, 11]
#queries = [8]

query_source_mapping = {
    1: ["bid"],
    2: ["bid"],
    3: ["auction", "person"],
    5: ["bid"],
    8: ["auction", "person"],
    11: ["bid"]
}
parallelisms = [1, 2, 4]
throughputs = [5000, 50000]

duration = 180

experiments = "query,throughput,parallelism,sources,partitions\n" # header

for q in queries:
    for p in parallelisms:
        for t in throughputs:
            if q in [1, 2, 3]:
                experiments += f"{q},{t*2},{p},{p},{p}\n"
            elif q == 8:
                experiments += f"{q},{t*3},{p},{p},{p}\n"
            else:
                experiments += f"{q},{t},{p},{p},{p}\n"
    
experiments=io.StringIO(experiments) # Convert to StringIO object

df = pd.read_csv(experiments) # Convert to Panda DataFrame

today = str(datetime.date.today())

df['date'] = today
df['exp_id'] = [str(uuid.uuid4()) for _,_ in df.iterrows()]
df['notebook_dir'] = ["%s/experiments/%s/%s/%s/%s" % (cur_dir, today, notebook_id, "Q"+str(i['query']), i['exp_id']) for _,i in df.iterrows()]
df['state'] = 'created'

# randomize test execution
pathlib.Path("%s/experiments/%s/%s/" % (cur_dir, today,notebook_id)).mkdir(exist_ok=True, parents=True)
df.to_csv("%s/experiments/%s/%s/experiment.csv" % (cur_dir, today, notebook_id))
df

## 3. Run Experiments

Iterates over the experiment matrix and executes `single_experiment.ipynb` for each
configuration via papermill. Already-executed runs (state `'executed'`) are skipped,
making the loop **resumable** after interruption.

For each run:
1. Creates the output directory and copies `config/` into it
2. Executes `single_experiment.ipynb` with injected parameters, saving the executed
   notebook as `Q<query>-<throughput>-<parallelism>.ipynb` in the run directory
3. Updates `experiment.csv` with the run state (`executed` or `error`)

A `KeyboardInterrupt` aborts the campaign cleanly without marking the current run as an error.

In [ ]:
import papermill as pm
import pathlib
import shutil

# read the description of the experiment
df = pd.read_csv("%s/experiments/%s/%s/experiment.csv" % (cur_dir, today, notebook_id))
for index, exp in df.iterrows():
    if exp['state'] == 'executed':# or exp["query"] == 1 or exp["parallelism"] == 4:
        continue
    print("Running experiment number: %d, ID: %s, query: %s,throughput: %s, parallelism %f" % (index, exp['exp_id'], exp['query'], exp['throughput'], exp['parallelism']))
    pathlib.Path(exp['notebook_dir']).mkdir(exist_ok=True, parents=True)
    pathlib.Path(f"{exp['notebook_dir']}/config").mkdir(exist_ok=True, parents=True)

    shutil.copytree(f"{cur_dir}/config",f"{exp['notebook_dir']}/config", dirs_exist_ok=True)

    try:
        pm.execute_notebook(
            'single_experiment.ipynb',
            '%s/Q%s-%s-%s.ipynb' % (exp['notebook_dir'], exp['query'], exp['throughput'], exp['parallelism']),
            cwd = exp['notebook_dir'],  # going to folder to execute notebook
            parameters = dict(
                output_dir = exp['notebook_dir'],
                query = f"q{exp['query']}.sql",
                sources = query_source_mapping[exp["query"]],
                throughput = exp['throughput'],
                parallelism = exp['parallelism'],
                init_monitoring = False,
                nb_partitions = exp['partitions'],
                duration = duration,
                cur_dir = cur_dir,
                max_parallelism=max_parallelism,
                cluster=cluster
            )
        )
    except KeyboardInterrupt:
        assert(False)
    except:
        df.at[index, 'state'] = 'error'
    else:
        df.at[index, 'state'] = 'executed'
        

    df.to_csv("%s/experiments/%s/%s/experiment.csv" % (cur_dir, today, notebook_id))
    

## 4. Generate Plots

Copies `plot.ipynb` into the experiment folder and executes it via papermill.
The executed notebook is saved as `plot_executed.ipynb` alongside `experiment.csv`.
`plot.ipynb` reads `experiment.csv` and the per-run result CSVs to produce comparison plots
across vanilla and Alloy for each query.

In [ ]:
import shutil

experiment_dir = "%s/experiments/%s/%s/" % (cur_dir, today, notebook_id)
shutil.copy(f"{cur_dir}/plot.ipynb", f"{experiment_dir}/plot.ipynb")
pm.execute_notebook(
    f"{cur_dir}/plot.ipynb",
    f"{experiment_dir}/plot_executed.ipynb",
    cwd=experiment_dir
)